In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
def get_bboxes(predictions, is_pred=True, num_classes=20, grid_size=7, num_boxes=2):
    """
    Convert the raw output of the YOLOv1 model into a list of bounding boxes with class labels and confidence scores.
    Parameters
    ----------
    predictions: Tensor of shape (batch_size, S, S, B*5 + C)
    is_pred: True if the input is model predictions, False if it's ground truth
    num_classes: Number of classes
    grid_size: Size of the grid (S)
    num_boxes: Number of bounding boxes per grid cell (B)

    Returns
    -------
    list of lists
        A list of bounding boxes for each image in the batch, where each bounding box is represented as a list [class_pred, prob_score, x1, y1, x2, y2].

    """
    batch_size = predictions.shape[0]

    # Split the output into bounding box coordinates and class probabilities
    box1 = predictions[..., 1:5]    # box1: [batch_size, S, S, 4] (x, y, w, h) for the first box
    box2 = predictions[..., 6:10]   # box2: [batch_size, S, S, 4] (x, y, w, h) for the second box


    if is_pred:
        score1 = predictions[..., 0:1]    # Confidence score for box1
        score2 = predictions[..., 5:6]    # Confidence score for box2
        best_box_mask = score1 >= score2
        
        best_boxes = best_box_mask * box1 + (~best_box_mask) * box2
        best_scores = best_box_mask * score1 + (~best_box_mask) * score2                # best_scores: P(object)
    else:
        # For ground truth, we can simply take the first box as the best box since there should only be one box per grid cell in the ground truth.
        best_boxes = box1
        best_scores = predictions[..., 0:1]
        
    # Get the class with the highest probability for each grid cell
    class_probs = predictions[..., 10:]
    best_class_prob, best_class_index = torch.max(class_probs, dim=-1, keepdim=True)    # best_class_prob: P(class|object), [batch_size, S, S, 1]; best_class_index: [batch_size, S, S, 1]

    # Convert the bounding box coordinates from (x, y, w, h) to (x1, y1, x2, y2)
    cell_indices = (torch.arange(grid_size).repeat(grid_size, 1)).unsqueeze(-1)         # [grid_size, grid_size, 1]
    x = 1 / grid_size * (best_boxes[..., 0:1] + cell_indices)
    y = 1 / grid_size * (best_boxes[..., 1:2] + cell_indices.permute(1, 0, 2))
    
    w = best_boxes[..., 2:3]
    h = best_boxes[..., 3:4]

    # best_scores * best_class_prob: P(object) * P(class|object) = P(class)
    converted_bboxes = torch.cat((best_class_index.float(), best_scores * best_class_prob, x, y, w, h), dim=-1)     # [class_pred, prob_score, x, y, w, h], shape: [batch_size, S, S, 6]

    # Flatten to a list for each batch
    all_bboxes = []
    for batch_idx in range(batch_size):
        batch_bboxes = []
        for i in range(grid_size):
            for j in range(grid_size):
                batch_bboxes.append([x.item() for x in converted_bboxes[batch_idx, i, j, :]])
        all_bboxes.append(batch_bboxes)

    return all_bboxes # shape: [batch_size, S*S, 6]


In [3]:
def iou(box1, box2):
    """
    Compute the Intersection over Union (IoU) of two bounding boxes.
    Supports batches of boxes with shape [..., 4] where format is [..., x, y, w, h].

    Parameters
    ----------
    box1 : Tensor of shape [..., 4] with format [..., x, y, w, h]
        The coordinates of the first bounding box(es).
    box2 : Tensor of shape [..., 4] with format [..., x, y, w, h]
        The coordinates of the second bounding box(es).

    Returns
    -------
    float
        The IoU of the two bounding boxes.
    """

    # Convert (x, y, w, h) to (x1, y1, x2, y2)
    x11 = box1[..., 0] - box1[..., 2] / 2
    y11 = box1[..., 1] - box1[..., 3] / 2
    x21 = box1[..., 0] + box1[..., 2] / 2
    y21 = box1[..., 1] + box1[..., 3] / 2

    x12 = box2[..., 0] - box2[..., 2] / 2
    y12 = box2[..., 1] - box2[..., 3] / 2
    x22 = box2[..., 0] + box2[..., 2] / 2
    y22 = box2[..., 1] + box2[..., 3] / 2

    # Compute the intersection coordinates  
    xA = torch.max(x11, x12)
    yA = torch.max(y11, y12)
    xB = torch.min(x21, x22)
    yB = torch.min(y21, y22)

    # Compute the area of intersection rectangle
    interArea = torch.clamp(xB - xA, min=0.0) * torch.clamp(yB - yA, min=0.0)

    # Compute the area of both the prediction and ground-truth rectangles
    box1Area = (x21 - x11) * (y21 - y11)
    box2Area = (x22 - x12) * (y22 - y12)

    # Compute the IoU
    iou = interArea / (box1Area + box2Area - interArea + 1e-6)

    return iou

In [4]:
def non_max_suppression(bboxes, iou_threshold, threshold):
    """
    Perform Non-Maximum Suppression (NMS) to eliminate redundant overlapping bounding boxes.
    Parameters
    ----------
    bboxes: list of lists in the format [class_pred, prob_score, x, y, w, h]
    iou_threshold: IoU threshold to suppress boxes
    threshold: minimum probability score to keep a box

    Returns
    -------
    list of lists
        The list of bounding boxes after applying NMS, in the same format as the input.
    """

    assert type(bboxes) == list

    # Filter out boxes with low confidence scores
    bboxes = [box for box in bboxes if box[1] > threshold]

    # Sort the boxes by probability score in descending order
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    bboxes_after_nms = []

    while bboxes:
        chosen_box = bboxes.pop(0)

        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0] # Ignore boxes of the same class
            or iou(torch.tensor(chosen_box[2:6]), torch.tensor(box[2:6])) < iou_threshold # Ignore boxes with IoU above the threshold
        ]

        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms

In [5]:
class YOLOLoss(nn.Module):
    def __init__(self, grid_size=7, num_boxes=2, num_classes=20, lambda_coord=5, lambda_noobj=0.5):
        """
        YOLOv1 Loss Function that combines localization loss, confidence loss, and classification loss.
        Parameters
        ----------
        grid_size: Grid size (number of cells along width and height)
        num_boxes: Number of bounding boxes predicted per grid cell
        num_classes: Number of classes
        lambda_coord: Weight for the localization loss
        lambda_noobj: Weight for the confidence loss for cells without objects
        """

        super().__init__()
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        self.num_classes = num_classes
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj

    def forward(self, predictions, target):
        """
        Compute the YOLOv1 loss.
        Parameters
        ----------
        predictions: Tensor of shape (batch_size, S, S, B*5 + C) - raw output from the model
        target: Tensor of shape (batch_size, S, S, B*5 + C) - ground truth labels

        Returns
        -------
        Tensor
            The computed loss value (scalar).
        """

        batch_size = predictions.size(0)

        box1_pred = predictions[..., 0:5]                                                                                       # [batch_size, S, S, 5]
        box2_pred = predictions[..., 5:10]                                                                                      # [batch_size, S, S, 5]
        tgt_box = target[..., 0:5]                                                                                              # [batch_size, S, S, 5]

        # Calculate IoU only on box coordinates (x, y, w, h), excluding confidence
        iou_box1 = iou(box1_pred[..., 1:5], tgt_box[..., 1:5])                                                                  # [batch_size, S, S]
        iou_box2 = iou(box2_pred[..., 1:5], tgt_box[..., 1:5])                                                                  # [batch_size, S, S]

        # Determine which box is responsible for the object (the one with the highest IoU)
        best_box_mask = (iou_box1 > iou_box2).unsqueeze(-1).float()                                                             # [batch_size, S, S, 1]

        exist_box = (target[..., 0:1] > 0).float()                                                                              # [batch_size, S, S, 1]

        # STEP 1: Localization loss (only for the responsible box)
        box_pred = best_box_mask * box1_pred + (1 - best_box_mask) * box2_pred                                                  # [batch_size, S, S, 5]
        loc_loss = self.lambda_coord * torch.sum(exist_box * torch.pow(box_pred[..., 1:5] - tgt_box[..., 1:5], 2))
        
        # STEP 2: Size loss (only for the responsible box)
        size_loss = self.lambda_coord * torch.sum(exist_box * torch.pow(torch.sqrt(torch.abs(box_pred[..., 3:5])) - torch.sqrt(torch.abs(tgt_box[..., 3:5])), 2))

        # STEP 3: Confidence loss
        conf_pred = best_box_mask * box1_pred[..., 0:1] + (1 - best_box_mask) * box2_pred[..., 0:1]                             # [batch_size, S, S, 1]
        conf_loss_obj = torch.sum(exist_box * torch.pow(conf_pred - tgt_box[..., 0:1], 2))
        conf_loss_noobj = self.lambda_noobj * torch.sum((1 - exist_box) * torch.pow(conf_pred - tgt_box[..., 0:1], 2))

        # STEP 4: Classification loss (only for cells with objects)
        class_loss = torch.sum(exist_box * torch.pow(predictions[..., 10:] - target[..., 10:], 2))
        
        # STEP 5: Total loss
        total_loss = (loc_loss + size_loss + conf_loss_obj + conf_loss_noobj + class_loss) / batch_size

        return total_loss

In [6]:
class YOLOv1(nn.Module):
    def __init__(self, num_classes=20, grid_size=7, num_boxes=2):
        """
        YOLOv1 architecture based on the original paper by Joseph Redmon et al. (2016).
         - Input: 448x448 RGB image
         - Output: SxSx(B*5 + C) tensor, where S=7, B=2, C=20 for VOC dataset
         - Convolutional layers followed by fully connected layers to predict bounding boxes and class probabilities.
         - Uses Leaky ReLU activations and max pooling for downsampling.
         - The final output is reshaped to [batch_size, S, S, B*5 + C] for easier interpretation of predictions.
         - The architecture can be further optimized or modified for different datasets or requirements, but this serves as a solid baseline implementation of YOLOv1.

        Parameters
        ----------
        num_classes: The number of object classes to predict.
        grid_size: The size of the grid (S) that divides the input image.
        num_boxes: The number of bounding boxes (B) predicted per grid cell.
        """

        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        
        # Define the convolutional layers
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),                                       # [batch_size, 64, 224, 224]
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),                                                      # [batch_size, 64, 112, 112]
            
            nn.Conv2d(64, 192, kernel_size=3, padding=1),                                               # [batch_size, 192, 112, 112]
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),                                                      # [batch_size, 192, 56, 56]
            
            nn.Conv2d(192, 128, kernel_size=1),                                                         # [batch_size, 128, 56, 56]
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),                                              # [batch_size, 256, 56, 56]
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=1),                                                         # [batch_size, 256, 56, 56]
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),                                              # [batch_size, 512, 56, 56]
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),                                                      # [batch_size, 512, 28, 28]
            
            nn.Conv2d(512, 256, kernel_size=1),                                                         # [batch_size, 256, 28, 28] x 4
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),          
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 256, kernel_size=1),                     
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),          
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=1),                     
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),                                             # [batch_size, 1024, 28, 28]
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2, stride=2),                                                      # [batch_size, 1024, 14, 14]

            nn.Conv2d(1024, 512, kernel_size=1),                                                        # [batch_size, 512, 14, 14] x 2
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 512, kernel_size=1),                    
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1), 
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),                                            # [batch_size, 1024, 14, 14]
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),                                  # [batch_size, 1024, 7, 7]

            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),                                            # [batch_size, 1024, 7, 7] x 2
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1)
        )
        
        # Fully connected layers
        self.output_size = self.grid_size * self.grid_size * (5 * self.num_boxes + self.num_classes)    # 7*7*(5*2+20) = 1470
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                                                                               # [batch_size, 1024 * 7 * 7]
            nn.Linear(1024 * self.grid_size * self.grid_size, 4096),
            nn.LeakyReLU(0.1),
            nn.Linear(4096, self.output_size)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        
        # Output structure: [Confidence score, class probabilities, x1, y1, x2, y2] for each box + class probabilities
        return x.view(-1, self.grid_size, self.grid_size, 5 * self.num_boxes + self.num_classes)        # [batch_size, 7, 7, 30]
        

    def backward(self, y_pred, y_true, loss_fn):
        """
        Compute the backward pass and update the model parameters based on the computed loss.
        Parameters
        ----------
        y_pred: The predicted output from the forward pass (raw output from the model).
        y_true: The ground truth labels corresponding to the input images.
        loss_fn: The loss function to compute the loss value (e.g., YOLOLoss).

        Returns
        -------
        Tensor
            The computed loss value for the current batch.
        """

        # Compute the loss using the provided loss function
        loss = loss_fn(y_pred, y_true)

        # Backpropagate the loss and update model parameters
        loss.backward()

        return loss.item()
    
    def fit(self, X, y, epochs=10, batch_size=32, lr=0.001, print_every=1):
        """
        Train the YOLOv1 model on the provided dataset.
        Parameters
        ----------
        X: Tensor of shape (num_samples, 3, 448, 448) - input images
        y: Tensor of shape (num_samples, S, S, B*5 + C) - ground truth labels
        epochs: Number of training epochs
        batch_size: Size of each training batch
        lr: Learning rate for the optimizer
        print_every: Frequency of printing training progress

        Returns
        -------
        None
        """

        # Initialize the loss function and optimizer
        loss_fn = YOLOLoss(grid_size=self.grid_size, num_boxes=self.num_boxes, num_classes=self.num_classes)
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)

        # Create a data loader for batching
        dataset = torch.utils.data.TensorDataset(X, y)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

        for epoch in range(epochs):
            self.train()  # Set the model to training mode
            optimizer.zero_grad()

            # Iterate over batches
            for batch_X, batch_y in dataloader:
                # Forward pass
                y_pred = self.forward(batch_X)
                loss = self.backward(y_pred, batch_y, loss_fn)

            # Update model parameters
            optimizer.step()

            if (epoch + 1) % print_every == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss:.4f}")

In [7]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 448
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

VOC_CLASSES = (
    "aeroplane", "bicycle", "bird", "boat",
    "bottle", "bus", "car", "cat", "chair",
    "cow", "diningtable", "dog", "horse",
    "motorbike", "person", "pottedplant",
    "sheep", "sofa", "train", "tvmonitor"
)
class_to_idx = {cls: idx for idx, cls in enumerate(VOC_CLASSES)}

In [8]:
def encode_target(boxes, grid_size=7, num_boxes=2, num_classes=20):
    """
    Encode the bounding box information into a target tensor suitable for training the YOLOv1 model.
    Parameters
    ----------
    boxes: Tensor of shape (num_boxes, 5) where each box is represented as [class_id, cx, cy, w, h]
    grid_size: The size of the grid (S) that divides the input image
    num_boxes: The number of bounding boxes (B) predicted per grid cell
    num_classes: The number of classes (C)

    Returns
    -------
    Tensor
        A tensor of shape (S, S, B*5 + C) that encodes the bounding box information 
        and class labels for each grid cell, suitable for training the YOLOv1 model.    
    """
    
    target = torch.zeros(grid_size, grid_size, num_boxes * 5 + num_classes)

    for box in boxes:
        class_id = int(box[0])
        cx, cy, w, h = box[1], box[2], box[3], box[4]
        
        col = int(cx * grid_size)  # j (x axis)
        row = int(cy * grid_size)  # i (y axis)
        col = min(col, grid_size - 1)
        row = min(row, grid_size - 1)
        
        if target[row, col, 0] == 0:
            # Box 1: [conf, x_cell, y_cell, w, h]
            target[row, col, 0] = 1                     # conf = 1
            target[row, col, 1] = cx * grid_size - col  # x relative to cell
            target[row, col, 2] = cy * grid_size - row  # y relative to cell
            target[row, col, 3] = w
            target[row, col, 4] = h
            # Class one-hot
            target[row, col, 10 + class_id] = 1

    return target  # [S, S, 30]

class CustomDataset(torchvision.datasets.VOCDetection):
    def __init__(self, root, year='2012', image_set='train', download=False, transform=None):
        super().__init__(root, year, image_set, download, transform)
    def __getitem__(self, index):
        image, target = super().__getitem__(index)

        # Size of the original image (before resizing)
        width = int(target['annotation']['size']['width'])
        height = int(target['annotation']['size']['height'])
        
        boxes = []
        
        # Convert bounding box coordinates to YOLO format (class_id, cx, cy, w, h)
        for obj in target['annotation']['object']:
            class_name = obj['name'].lower().strip()
            if class_name not in class_to_idx:
                continue
            class_id = class_to_idx[class_name]
            
            bndbox = obj['bndbox']
            xmin = float(bndbox['xmin'])
            ymin = float(bndbox['ymin'])
            xmax = float(bndbox['xmax'])
            ymax = float(bndbox['ymax'])
            
            cx = ((xmin + xmax) / 2.0) / width
            cy = ((ymin + ymax) / 2.0) / height
            w = (xmax - xmin) / width
            h = (ymax - ymin) / height
            
            boxes.append([class_id, cx, cy, w, h])
            
        target_tensor = torch.tensor(boxes, dtype=torch.float32)
        target_tensor = encode_target(target_tensor)
        
        return image, target_tensor

def yolovoc_collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, 0)
    targets = torch.stack(targets, 0)  # Stack targets into a batch tensor [batch_size, 7, 7, 30]
    return images, targets

dataset = CustomDataset(root='./data', year='2012', image_set='train', download=False, transform=transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=yolovoc_collate_fn)

# Get first batch
for images, targets in dataloader:
    print("Batch of images shape:", images.shape)  # Should be [batch_size, 3, 448, 448]
    print("Batch of targets shape:", targets.shape) # Should be [batch_size, 7, 7, 30]
    break

Batch of images shape: torch.Size([8, 3, 448, 448])
Batch of targets shape: torch.Size([8, 7, 7, 30])


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = YOLOv1(num_classes=20, grid_size=7, num_boxes=2)
model.to(device)

model.fit(X=images, y=targets, epochs=10, batch_size=8, lr=0.0001, print_every=1)
torch.save(model.state_dict(), 'yolov1.pth')

Epoch [1/10], Loss: 37.1537
Epoch [2/10], Loss: 32.6765
Epoch [3/10], Loss: 27.3585
Epoch [4/10], Loss: 27.6682
Epoch [5/10], Loss: 20.5499
Epoch [6/10], Loss: 19.6335
Epoch [7/10], Loss: 17.2608
Epoch [8/10], Loss: 14.4899
Epoch [9/10], Loss: 16.5512
Epoch [10/10], Loss: 14.2948
